In [ ]:
import pandas as pd
import json
import random
import os

# 1. DOSSIERS
os.makedirs("data/processed", exist_ok=True)

# 2. CHARGEMENT DE TON LEXIQUE (522 MOTS)
chatbot_data = []
try:
    local_df = pd.read_csv('mon_lexique.csv')
    c1, c2 = local_df.columns[0], local_df.columns[1]
    for _, row in local_df.iterrows():
        chatbot_data.append({
            "instruction": f"Comment dit-on en Dioula : {row[c1]} ?",
            "response": f"En Dioula, on dit : {row[c2]}."
        })
    print(f"✅ Lexique local : {len(chatbot_data)} mots ajoutés.")
except Exception as e:
    print(f"❌ Erreur locale : {e}")

# 3. CHARGEMENT MASAKHANE (Format Parquet - Le plus fiable)
print("--- Tentative Masakhane via Parquet ---")
try:
    # URL vers le stockage Parquet de Hugging Face pour fra-bam
    url_parquet = "https://huggingface.co/datasets/masakhane/mafand/resolve/main/data/fra-bam/train.parquet"
    df_m = pd.read_parquet(url_parquet)

    count_m = 0
    for _, row in df_m.iterrows():
        # On extrait selon la structure Parquet (souvent une colonne 'translation' contenant un dict)
        if 'translation' in row:
            f = row['translation']['fra']
            b = row['translation']['bam']
        else:
            f, b = row['source_sentence'], row['target_sentence']

        chatbot_data.append({
            "instruction": f"Comment dit-on en Dioula : {f} ?",
            "response": f"En Dioula, on dit : {b}."
        })
        count_m += 1
    print(f"✅ Masakhane : {count_m} phrases ajoutées.")
except Exception as e:
    print(f"⚠️ Échec Parquet : {e}. On tente le CSV brut...")
    try:
        url_csv = "https://huggingface.co/datasets/masakhane/mafand/resolve/main/data/fra-bam/train.csv"
        df_c = pd.read_csv(url_csv)
        for _, row in df_c.iterrows():
            chatbot_data.append({"instruction": f"Comment dit-on en Dioula : {row[0]} ?", "response": f"En Dioula, on dit : {row[1]}."})
        print(f"✅ Masakhane (via CSV) ajouté !")
    except:
        print("❌ Masakhane indisponible. On reste sur tes 522 mots.")

# 4. SAUVEGARDE
if chatbot_data:
    random.shuffle(chatbot_data)
    with open("data/processed/final_chatbot_data.jsonl", "w", encoding="utf-8") as f:
        for entry in chatbot_data:
            json.dump(entry, f, ensure_ascii=False)
            f.write("\n")
    print(f"\n🚀 TOTAL FINAL : {len(chatbot_data)} lignes prêtes !")

In [ ]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration

# 1. Initialisation du Tokenizer
model_id = "google/flan-t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_id)

# 2. Fonction de prétraitement (Correction de l'indentation)
def preprocess_function(examples):
    # Tout ce qui est ici doit être décalé d'un cran (4 espaces)
    model_inputs = tokenizer(
        examples["instruction"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=examples["response"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# 3. Application au dataset (On s'assure que 'dataset' existe encore)
try:
    tokenized_dataset = dataset.map(preprocess_function, batched=True)
    print("🚀 Dataset prêt pour l'entraînement !")
except NameError:
    print("❌ Erreur : Le 'dataset' a disparu de la mémoire. Relance la cellule précédente qui crée le fichier JSONL.")

In [ ]:
import os
import pandas as pd
import json
import torch
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, DataCollatorForSeq2Seq, TrainingArguments, Trainer

# --- 1. RE-CRÉATION DU DOSSIER ET DU FICHIER ---
os.makedirs("data/processed", exist_ok=True)
chatbot_data = []

try:
    # Lecture directe de ton CSV
    local_df = pd.read_csv('mon_lexique.csv')
    c1, c2 = local_df.columns[0], local_df.columns[1]
    for _, row in local_df.iterrows():
        chatbot_data.append({
            "instruction": f"Comment dit-on en Dioula : {row[c1]} ?",
            "response": f"En Dioula, on dit : {row[c2]}."
        })

    # Sauvegarde forcée en JSONL
    output_path = "data/processed/final_chatbot_data.jsonl"
    with open(output_path, "w", encoding="utf-8") as f:
        for entry in chatbot_data:
            json.dump(entry, f, ensure_ascii=False)
            f.write("\n")
    print(f"✅ Lexique restauré : {len(chatbot_data)} lignes.")
except Exception as e:
    print(f"❌ ERREUR CRITIQUE : Vérifie que 'mon_lexique.csv' est bien uploadé ! ({e})")

# --- 2. PRÉPARATION DU DATASET ET DU MODÈLE ---
if len(chatbot_data) > 0:
    model_id = "google/flan-t5-base"
    tokenizer = T5Tokenizer.from_pretrained(model_id)
    model = T5ForConditionalGeneration.from_pretrained(model_id)

    def preprocess_function(examples):
        model_inputs = tokenizer(examples["instruction"], max_length=128, truncation=True, padding="max_length")
        labels = tokenizer(text_target=examples["response"], max_length=128, truncation=True, padding="max_length")
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    dataset = load_dataset('json', data_files=output_path, split='train')
    tokenized_dataset = dataset.map(preprocess_function, batched=True)

  # --- 3. ENTRAÎNEMENT OPTIMISÉ ---
training_args = TrainingArguments(
    output_dir="./kan_chatbot_model",
    eval_strategy="no",
    learning_rate=1e-4,         # On augmente la vitesse (1e-4 au lieu de 5e-5)
    per_device_train_batch_size=4, # Batch plus petit pour des mises à jour plus fréquentes
    gradient_accumulation_steps=2,
    num_train_epochs=30,        # On double les époques pour forcer la mémorisation
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    logging_steps=10,           # On veut voir l'évolution tous les 10 pas
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    # On revient à la méthode la plus stable pour passer le tokenizer
)
# On injecte manuellement le tokenizer dans le trainer pour la compatibilité
trainer.tokenizer = tokenizer

print("🚀 Lancement de l'entraînement correctif...")
trainer.train()